In [2]:
from pennylane.fermi import FermiSentence, from_string
import pennylane as qml
from pennylane import numpy as np
import time
qml.about()

Name: pennylane
Version: 0.43.1
Summary: PennyLane is a cross-platform Python library for quantum computing, quantum machine learning, and quantum chemistry. Train a quantum computer the same way as a neural network.
Home-page: 
Author: 
Author-email: 
License-Expression: Apache-2.0
Location: /home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages
Requires: appdirs, autograd, autoray, cachetools, diastatic-malt, networkx, numpy, packaging, pennylane-lightning, requests, rustworkx, scipy, tomlkit, typing_extensions
Required-by: pennylane_lightning, pennylane_lightning_gpu

Platform info:           Linux-6.11.0-26-generic-x86_64-with-glibc2.39
Python version:          3.12.12
Numpy version:           2.3.4
Scipy version:           1.16.2
JAX version:             0.6.2
Installed devices:
- default.clifford (pennylane-0.43.1)
- default.gaussian (pennylane-0.43.1)
- default.mixed (pennylane-0.43.1)
- default.qubit (pennylane-0.43.1)
- default.qutrit (pennylane-0.43.1)
- default.qutrit.mix

In [3]:
# 1. 导入必要库（和之前一致，必须用 scipy.sparse）
import scipy.sparse as sp

# 2. 直接写文件名（相对路径，同目录下无需加任何路径前缀）
matrix_file = "L=6 n=3.npz"  # 重点：只写文件名，不用加 /Users/... 之类的路径

# 3. 加载矩阵（一行搞定，自动恢复 CSR 格式）
H_3 = sp.load_npz(matrix_file)

# 4. 验证加载成功（可选，快速确认没出错）
print("✅ 矩阵调用成功！")
print(f"矩阵格式：{H_3.format}")  # 输出 'csr'（和保存时一致）
print(f"矩阵形状：{H_3.shape}")    # 输出原矩阵的行数×列数
print(f"非零元素个数：{H_3.nnz}")  # 输出非零元素数量

from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
H= H_3.toarray()
# 计算模最小的特征值（注意可能为复数）
min_eigval = eigsh(H, k=2, which='SA', return_eigenvectors=True,)

min =  eigh( H,eigvals_only=True)[0]
print("最小特征值:", min_eigval[0])
print(min)

✅ 矩阵调用成功！
矩阵格式：csr
矩阵形状：(309, 309)
非零元素个数：7765
最小特征值: [-8.48179737  4.02035168]
-8.481797373159006


In [4]:

def get_Hami(H):
    """
    将任意维度的哈密顿量矩阵扩展为 2^Nq 维度（量子比特系统希尔伯特空间维度）

    参数:
        H: 原哈密顿量矩阵（quspin 哈密顿量或 numpy 数组，维度为 d×d）
    返回:
        Hami: 扩展后的哈密顿量矩阵（维度 l×l，l=2^Nq）
        Nq: 所需量子比特数（满足 2^Nq ≥ d 的最小整数）
    说明:
        量子比特系统的希尔伯特空间维度必为 2 的幂次，原玻色子基矢空间维度 Ns 可能非 2^Nq，
        故通过补零扩展至最近的 2^Nq 维度，确保与量子计算框架兼容
    """
    # 若输入为 quspin 哈密顿量，先转换为 numpy 数组（优先保留稀疏性，此处暂转 dense 便于理解）
    if hasattr(H, 'toarray'):
        H_dense = H.toarray()
    else:
        H_dense = np.asarray(H)

    d = H_dense.shape[0]  # 原哈密顿量维度（即玻色子基矢空间维度 Ns）
    Nq = int(np.ceil(np.log2(d)))  # 满足 2^Nq ≥ d 的最小量子比特数
    l = 2 ** Nq  # 扩展后的维度（量子比特系统维度）

    # 初始化扩展矩阵（补零）
    Hami = np.zeros((l, l), dtype=H_dense.dtype)
    Hami[:d, :d] = H_dense  # 原矩阵嵌入扩展矩阵的左上角，其余位置补零

    return Hami, Nq

Hami ,n_qubits  = get_Hami(H)
print(n_qubits)
H_decomp = qml.pauli_decompose(Hami)

9


In [5]:
import pennylane.optimize as opt
print([x for x in dir(opt) if x.endswith('Optimizer')])


['AdagradOptimizer', 'AdamOptimizer', 'AdaptiveOptimizer', 'GradientDescentOptimizer', 'MomentumOptimizer', 'MomentumQNGOptimizer', 'NesterovMomentumOptimizer', 'QNGOptimizer', 'QNSPSAOptimizer', 'RMSPropOptimizer', 'RiemannianGradientOptimizer', 'RotoselectOptimizer', 'RotosolveOptimizer', 'SPSAOptimizer', 'ShotAdaptiveOptimizer']


In [1]:
import pennylane as qml
import time
from pennylane import numpy as np
from scipy.optimize import minimize

# ===================  超参数  ===================
n_qubits = 9
depth    = 30
steps    = 1000          # 仅用于 PennyLane 优化器
seed     = 42
np.random.seed(seed)

# ===================  设备  ===================
dev = qml.device("lightning.gpu", wires=n_qubits)   # 没 GPU 就换 lightning.qubit

# ===================  哈密顿量  ===================


# ===================  Ansatz  ===================
def ansatz(params):
    params = params.reshape((depth, n_qubits))
    for d in range(depth):
        for i in range(n_qubits):
            qml.RY(params[d, i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

@qml.qnode(dev)
def energy_fn(params):
    ansatz(params)
    return qml.expval(qml.Hermitian(Hami, wires=range(n_qubits)))

# ---------- 扁平接口，供 scipy 使用 ----------
def flat_energy(p_flat):
    return float(energy_fn(p_flat))

def flat_grad(p_flat):
    return qml.grad(energy_fn)(p_flat)

# ===================  优化器列表  ===================
# 对 PennyLane 优化器用 step_and_cost；对 SLSQP 用 minimize
optimizers = {
    #"Adam" : (qml.optimize.AdamOptimizer, 0.05, {}),
    #"SPSA" : (qml.optimize.SPSAOptimizer, 0.02, {"c": 0.3}),
    "SLSQP": None,   # 特殊处理
}

# ===================  训练循环  ===================
results = {}
for name in optimizers:
    print(f"\n>>> 开始 {name} 优化")
    x0 = np.zeros(depth*n_qubits, requires_grad=True)

    tic = time.time()
    if name == "SLSQP":
        res = minimize(flat_energy, x0=x0, method="SLSQP",
                       options={"maxiter": 1000, "ftol": 1e-8})
        final_e, params = res.fun, res.x
    else:
        OptClass, lr, kw = optimizers[name]
        opt = OptClass(lr, **kw)
        for step in range(steps + 1):
            x0, loss = opt.step_and_cost(energy_fn, x0)
            if step % 100 == 0:
                print(f"  step {step:4d}  energy = {loss:.8f}")
        final_e = energy_fn(x0)
        params = x0
    toc = time.time()

    results[name] = final_e
    print(f"{name:10s} 最终能量 = {final_e:.8f}  耗时 = {toc-tic:.2f}s")

# ===================  汇总  ===================
print("\n" + "="*50)
for name, e in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:10s}  {e:.8f}")



>>> 开始 SLSQP 优化


NameError: name 'Hami' is not defined

In [16]:
import pennylane as qml
import time
from pennylane import numpy as np

# ===================  超参数  ===================
n_qubits = 9
depth    = 30
steps    = 1000
seed     = 42
np.random.seed(seed)



# ===================  设备  ===================
dev = qml.device("lightning.gpu", wires=n_qubits)

# ===================  Ansatz  ===================
def ansatz(params):
    params = params.reshape((depth, n_qubits))
    for d in range(depth):
        for i in range(n_qubits):

            qml.RY(params[d, i], wires=i)

        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

@qml.qnode(dev, interface="autograd")
def energy_fn(params):
    ansatz(params)
    return qml.expval(qml.Hermitian(Hami, wires=range(n_qubits)) )

# ===================  优化器列表  ===================
optimizers = {
    "Adam"     : (qml.optimize.AdamOptimizer, 0.05, {}),

}

# ===================  训练循环  ===================
results = {}
for name, (OptClass, lr, kw) in optimizers.items():
    print(f"\n>>> 开始 {name} 优化（共 {steps} 步）")
    params = np.random.uniform(0, 2*np.pi, (depth,  n_qubits), requires_grad=True)
    opt    = OptClass(lr, **kw) if lr is not None else OptClass(**kw)

    tic = time.time()
    for step in range(steps + 1):
        params, loss = opt.step_and_cost(energy_fn, params)
        if step % 50 == 0:
            print(f"  step {step:3d}  energy = {loss:.8f}")
    toc = time.time()

    final_e = energy_fn(params)
    results[name] = final_e
    print(f"{name:10s} 最终能量 = {final_e:.8f}  耗时 = {toc-tic:.2f}s")

# ===================  汇总  ===================
print("\n" + "="*50)
for name, e in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:10s}  {e:.8f}")



>>> 开始 Adam 优化（共 1000 步）
  step   0  energy = 325.47042508


/home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages/pennylane/devices/preprocess.py:283: UserWarning: Differentiating with respect to the input parameters of Hermitian is not supported with the adjoint differentiation method. Gradients are computed only with regards to the trainable parameters of the circuit.

 Mark the parameters of the measured observables as non-trainable to silence this warning.
  warnings.warn(


  step  50  energy = 6.82031491
  step 100  energy = 1.78943684
  step 150  energy = 1.06401031
  step 200  energy = 0.53378903
  step 250  energy = 0.82205835
  step 300  energy = -0.02465652
  step 350  energy = -0.23486345
  step 400  energy = -0.17224587
  step 450  energy = -0.36602649
  step 500  energy = -0.22252784
  step 550  energy = -0.18150445
  step 600  energy = -0.65978463
  step 650  energy = -0.37010880
  step 700  energy = -0.96309716
  step 750  energy = -1.00393898
  step 800  energy = -0.85462033
  step 850  energy = -1.04016198
  step 900  energy = -0.91001822
  step 950  energy = -1.14891742
  step 1000  energy = -1.13488827
Adam       最终能量 = -1.15989885  耗时 = 89.55s

>>> 开始 Nesterov 优化（共 1000 步）
  step   0  energy = 279.90066637
  step  50  energy = 285.92615346
  step 100  energy = 293.64842861
  step 150  energy = 276.98919338
  step 200  energy = 283.50949565
  step 250  energy = 299.73171143
  step 300  energy = 290.01218232
  step 350  energy = 329.84851805

In [8]:
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import AdamOptimizer,RotosolveOptimizer

##############################
# 1. 问题参数：比特数、HEA 深度
##############################
n_qubits = 9
N = n_qubits             # 任意 N 比特
depth = 20         # HEA 层数
steps = 400        # VQE 迭代步数
lr = 0.05          # Adam 学习率

##############################
# 2. 定义哈密顿量（Transverse-Field Ising）
##############################
         # hx

##############################
# 3. 构造 Hardware-Efficient Ansatz
#    每层：单比特旋转 RX-RY-RX + CNOT 纠缠
##############################
def hea_layer(params):
    """单个 HEA 层：params 形状 (3, N)"""
    for i in range(N):
        qml.RX(params[0, i], wires=i)
        qml.RY(params[1, i], wires=i)
        qml.RX(params[2, i], wires=i)
    # 线性邻接 CNOT 纠缠
    for i in range(N - 1):
        qml.CNOT(wires=[i, i + 1])

def ansatz(params):
    """depth 层 HEA"""
    params = params.reshape((depth, 3, N))
    for d in range(depth):
        hea_layer(params[d])

##############################
# 4. 定义 QNode：返回能量期望值
##############################
dev = qml.device("lightning.gpu", wires=N)

@qml.qnode(dev, interface="autograd")
def energy_fn(params):
    ansatz(params)
    return qml.expval(qml.Hermitian(Hami, wires=range(n_qubits)))

##############################
# 5. 初始化参数并运行 VQE
##############################
params = np.zeros((depth, 3, N), requires_grad=True)

opt = qml.optimize.NesterovMomentumOptimizer(0.01, 0.9)

print("开始 VQE 优化...")
for step in range(steps + 1):
    params, loss = opt.step_and_cost(energy_fn, params)
    if step % 10 == 0:
        print(f"Step {step:3d} | energy = {loss:.8f}")

print("\n最终基态能量估计:", energy_fn(params))

##############################
# 6. 可选：打印最终线路
##############################
# 在 PennyLane 0.43 里，只要这样即可：
# print(qml.draw(energy_fn)(params))




开始 VQE 优化...


/home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages/pennylane/devices/preprocess.py:283: UserWarning: Differentiating with respect to the input parameters of Hermitian is not supported with the adjoint differentiation method. Gradients are computed only with regards to the trainable parameters of the circuit.

 Mark the parameters of the measured observables as non-trainable to silence this warning.
  warnings.warn(


Step   0 | energy = 158.61996860
Step  10 | energy = 278.94260903
Step  20 | energy = 301.20278286
Step  30 | energy = 271.62135147
Step  40 | energy = 294.62207061
Step  50 | energy = 286.55301048
Step  60 | energy = 274.24254031
Step  70 | energy = 258.96802701
Step  80 | energy = 246.30009018
Step  90 | energy = 282.82622710
Step 100 | energy = 287.43333089
Step 110 | energy = 266.00112509
Step 120 | energy = 286.99735329
Step 130 | energy = 271.39026759
Step 140 | energy = 274.29386351
Step 150 | energy = 275.10371023
Step 160 | energy = 283.33472616
Step 170 | energy = 271.96100151
Step 180 | energy = 262.65462648
Step 190 | energy = 261.56188362
Step 200 | energy = 276.28697387
Step 210 | energy = 276.43448002
Step 220 | energy = 257.64500430
Step 230 | energy = 311.73511486
Step 240 | energy = 289.88335115
Step 250 | energy = 245.95355029
Step 260 | energy = 274.59478098
Step 270 | energy = 267.93197405
Step 280 | energy = 290.89061976
Step 290 | energy = 272.72965179
Step 300 |

In [ ]:
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import AdamOptimizer, RotosolveOptimizer

##############################
# 1. 问题参数：比特数、HEA 深度
##############################
n_qubits = 12
N = n_qubits             # 任意 N 比特
depth = 20               # HEA 层数
steps = 400              # VQE 迭代步数
lr = 0.05                # Adam 学习率（Rotosolve 不用）

##############################
# 2. 定义哈密顿量（Transverse-Field Ising）
##############################
# 这里你原来怎么定义 Hami 就怎么来
# Hami = ...

##############################
# 3. 构造 Hardware-Efficient Ansatz
##############################
def hea_layer(params):
    """单个 HEA 层：params 形状 (3, N)"""
    for i in range(N):
        qml.RX(params[0, i], wires=i)
        qml.RY(params[1, i], wires=i)
        qml.RX(params[2, i], wires=i)
    # 线性邻接 CNOT 纠缠
    for i in range(N - 1):
        qml.CNOT(wires=[i, i + 1])

def ansatz(params):
    """depth 层 HEA"""
    params = params.reshape((depth, 3, N))
    for d in range(depth):
        hea_layer(params[d])

##############################
# 4. 定义 QNode：返回能量期望值
##############################
dev = qml.device("lightning.gpu", wires=N)

@qml.qnode(dev, interface="autograd")
def energy_fn(params):
    ansatz(params)
    return qml.expval(qml.Hermitian(Hami, wires=range(n_qubits)))

##############################
# 5. 初始化参数
##############################
params = np.random.uniform(0, 2 * np.pi, size=(depth, 3, N), requires_grad=True)

##############################
# 5.1 为 Rotosolve 设置频率信息（关键！！）
##############################
# params 的名字就是 energy_fn 的参数名 "params"
# 形状 (depth, 3, N)，每个标量参数都只在一个 RX/RY/RX 旋转里，
# 对应的频率个数 = 1
nums_frequency = {
    "params": { (d, k, i): 1
                for d in range(depth)
                for k in range(3)
                for i in range(N) }
}

##############################
# 5.2 选择 Rotosolve 优化器
##############################
opt = qml.RotosolveOptimizer()

print("开始 VQE 优化（Rotosolve）...")
for step in range(steps + 1):
    params, loss = opt.step_and_cost(
        energy_fn,
        params,
        nums_frequency=nums_frequency,   # ⭐ 一定要传这一项
    )
    if step % 10 == 0:
        print(f"Step {step:3d} | energy = {loss:.8f}")

print("\n最终基态能量估计:", energy_fn(params))

##############################
# 6. 可选：打印最终线路
##############################
# print(qml.draw(energy_fn)(params))
